<a href="https://colab.research.google.com/github/HoriaCluj/Quant-ML-Projects/blob/main/Features_Dataset_Explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Features Dataset**

**`Aim`**: Updating the Order-Book State and storing its features (mid-price, imbalance, etc..) in a new dataset.

Each row of the new dataset will be given by sequentially updating the order-book state and extracting its features.

In [25]:
! pip install polars fastlob

In [26]:
%env FASTLOB_DECIMAL_PRECISION_PRICE=6
%env FASTLOB_DECIMAL_PRECISION_QTY=6

env: FASTLOB_DECIMAL_PRECISION_PRICE=6
env: FASTLOB_DECIMAL_PRECISION_QTY=6


Installing `polars` for dataframe creation & manipulation
and `fastlob` as an efficient way to work with orderbook data.

In [27]:
import polars as pl
from fastlob import Orderbook

Modify the following lines to your own system (I ran this on Google Colab):

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
df = pl.read_csv('/content/drive/MyDrive/Colab_Notebooks/Beta_Sigma/ETHBTC.csv')

The **`snapshot`** column indicates the type of entry:
- `snapshot = True` refers to a full order book state at a specific point in time (all bids & asks at a certain time `t`)
- `snapshot = False` represents an update in the order book, these are incremental changes to the existing order book state.

In [30]:
snapshot = df.filter((pl.col('snapshot') == True) & (pl.col('chunk') == 0))

In [7]:
snapshot

symbol,side,price,amount,snapshot,chunk,update_id,timestamp
str,str,f64,f64,bool,i64,i64,str
"""ETHBTC""","""bid""",0.02324,10.7012,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""bid""",0.02323,177.6961,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""bid""",0.02322,121.7039,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""bid""",0.02321,99.5636,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""bid""",0.0232,89.8007,true,0,0,"""2025-03-24T00:49:12.665787Z"""
…,…,…,…,…,…,…,…
"""ETHBTC""","""ask""",0.071661,0.0069,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""ask""",0.071664,0.002,true,0,0,"""2025-03-24T00:49:12.665787Z"""
"""ETHBTC""","""ask""",0.071665,0.0027,true,0,0,"""2025-03-24T00:49:12.665787Z"""


In the `ETHBTC` dataset we are using in this notebook, there are 52 "independent" chunks, the data is thus time-continuous only within a specific chunk

In [54]:
len(df['chunk'].unique())

52

The `sc` dictionnary stores the initial snapshot of the order book:
- The `bids` key holds a list of tuples with `(price, amount)` and represents a bid entry from the order-book snapshot.
- The `asks` key similarly holds a list of tuples with `(price, amount)` representing an ask entry from the order-book snapshot.

In [31]:
sc = {'bids': [], 'asks': []}

In [32]:
for x in snapshot.filter(pl.col('side') == 'bid').iter_rows():
    price, amount = x[2], x[3]
    sc['bids'].append((price, amount))

for x in snapshot.filter(pl.col('side') == 'ask').iter_rows():
    price, amount = x[2], x[3]
    sc['asks'].append((price, amount))

In [33]:
lob = Orderbook.from_snapshot(sc, start=True)
lob

Order-book LOB
- started=True
- running-time=0s
- #prices=6336
- #asks=5000
- #bids=1336

Displaying the current Order-Book state using `fastlob`

In [34]:
print(lob.view())

   [ORDER-BOOK LOB]

   ...(4990 more asks)
 - 0.023340 | 000 | 20.660500 | 0.482216070000
 - 0.023330 | 000 | 6.276100 | 0.146421413000
 - 0.023320 | 000 | 86.443800 | 2.015869416000
 - 0.023310 | 000 | 40.240100 | 0.937996731000
 - 0.023300 | 000 | 26.587500 | 0.619488750000
 - 0.023290 | 000 | 42.320200 | 0.985637458000
 - 0.023280 | 000 | 57.317500 | 1.334351400000
 - 0.023270 | 000 | 60.380100 | 1.405044927000
 - 0.023260 | 000 | 161.924200 | 3.766356892000
 - 0.023250 | 000 | 166.139400 | 3.862741050000
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 - 0.023240 | 000 | 10.701200 | 0.248695888000
 - 0.023230 | 000 | 177.696100 | 4.127880403000
 - 0.023220 | 000 | 121.703900 | 2.825964558000
 - 0.023210 | 000 | 99.563600 | 2.310871156000
 - 0.023200 | 000 | 89.800700 | 2.083376240000
 - 0.023190 | 000 | 78.633600 | 1.823513184000
 - 0.023180 | 000 | 7.689200 | 0.178235656000
 - 0.023170 | 000 | 24.509200 | 0.567878164000
 - 0.023160 | 000 | 163.987000 | 3.797938920000
 - 0.023150 |

Visualizing the `First Update` (100 ms)

In [43]:
display(U[1])

{'bids': [(0.02324, 10.7012),
  (0.02323, 107.7572),
  (0.02322, 187.1235),
  (0.02321, 99.7029),
  (0.0232, 94.3691),
  (0.02299, 5.8316)],
 'asks': [(0.02326, 159.9843), (0.02329, 42.3202)]}

In [36]:
updates = []

The `chunk0` variable stores a new DataFrame containing all the `updates` (`snapshot = False`) of `chunk 0`.

Remember that the original order-book state of `chunk0` is given by filtering the original df with `snapshot = True` and `chunk = 0`

In [37]:
chunk0 = df.filter((pl.col('chunk') == 0) & (pl.col('snapshot') == False))

In [38]:
max_update_id = 1000 # chunk0['update_id'].max()
max_update_id

1000

In [39]:
for update_id in range(1, max_update_id + 1):
    updates.append(df.filter((pl.col('chunk') == 0) & (pl.col('update_id') == update_id)))

In [40]:
U = []
for update in updates:
    u = {'bids': [], 'asks': []}

    for x in update.filter(pl.col('side') == 'bid').iter_rows():
        price, amount = x[2], x[3]
        u['bids'].append((price, amount))

    for x in update.filter(pl.col('side') == 'ask').iter_rows():
        price, amount = x[2], x[3]
        u['asks'].append((price, amount))

    U.append(u)

In [41]:
lob.load_updates(U)

**`lob.view()`** displays the current state of the order-book (changes every time an update is applied to the order-book state via `lob.step`)

**`lob.step()`** method applies the next available update (that has been loaded using `lob.load_updates()` to the current state of the order-book. It processes these updates sequentially, thus the order book moves forward in time

In [19]:
print(lob.view())
lob.step()

   [ORDER-BOOK LOB]

   ...(4990 more asks)
 - 0.023340 | 000 | 20.660500 | 0.482216070000
 - 0.023330 | 000 | 6.276100 | 0.146421413000
 - 0.023320 | 000 | 86.443800 | 2.015869416000
 - 0.023310 | 000 | 40.240100 | 0.937996731000
 - 0.023300 | 000 | 26.587500 | 0.619488750000
 - 0.023290 | 000 | 42.320200 | 0.985637458000
 - 0.023280 | 000 | 57.317500 | 1.334351400000
 - 0.023270 | 000 | 60.380100 | 1.405044927000
 - 0.023260 | 000 | 161.924200 | 3.766356892000
 - 0.023250 | 000 | 166.139400 | 3.862741050000
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 - 0.023240 | 000 | 10.701200 | 0.248695888000
 - 0.023230 | 000 | 177.696100 | 4.127880403000
 - 0.023220 | 000 | 121.703900 | 2.825964558000
 - 0.023210 | 000 | 99.563600 | 2.310871156000
 - 0.023200 | 000 | 89.800700 | 2.083376240000
 - 0.023190 | 000 | 78.633600 | 1.823513184000
 - 0.023180 | 000 | 7.689200 | 0.178235656000
 - 0.023170 | 000 | 24.509200 | 0.567878164000
 - 0.023160 | 000 | 163.987000 | 3.797938920000
 - 0.023150 |

#### Generating the **Dataframe with Features** for `chunk 0`

Each row represents the features of the current order-book state.

Thus every new row is the updated order-book state's features (100 milliseconds later).

In [55]:
new_lob = Orderbook.from_snapshot(sc, start=True)

features_data = []

# Function to Extract Features
def extract_features(lob_obj):
    return {
        'spread': lob_obj.spread(),
        'midprice': lob_obj.midprice(),
        'weighted midprice': lob_obj.weighted_midprice(),
        'imbalance': lob_obj.imbalance(),
        'asks_volume': lob_obj.asks_volume(),
        'bids_volume': lob_obj.bids_volume(),
        'total volume': lob_obj.total_volume()
    }

# Add features for the initial snapshot state
initial_features = extract_features(new_lob)
features_data.append(initial_features)

# Iterate through each update, apply it, and extract features
for i, update_dict in enumerate(U):
    new_lob.load_updates([update_dict]) # Load one update at a time
    new_lob.step() # Apply the loaded update
    current_features = extract_features(new_lob)
    features_data.append(current_features)

# Create a Polars DataFrame from the collected features
features_df = pl.DataFrame(features_data)
# Controlling the number of decimals in each
features_df = features_df.with_columns(
    pl.col('spread').cast(pl.Float64),
    pl.col('midprice').cast(pl.Float64),
    pl.col('weighted midprice').cast(pl.Float64),
    pl.col('imbalance').cast(pl.Float64),
    pl.col('asks_volume').cast(pl.Float64),
    pl.col('bids_volume').cast(pl.Float64),
    pl.col('total volume').cast(pl.Float64)
)

# Display the DataFrame with updated decimal precision
display(features_df.head())

spread,midprice,weighted midprice,imbalance,asks_volume,bids_volume,total volume
f64,f64,f64,f64,f64,f64,f64
0.00001,0.023245,0.023249,0.497507,23425.7576,23193.3354,46619.093
0.00001,0.023245,0.023249,0.497507,23425.7576,23193.3354,46619.093
0.00001,0.023245,0.023249,0.49753,23423.8177,23193.5238,46617.3415
0.00001,0.023245,0.023249,0.497589,23423.8177,23199.008,46622.8257
0.00001,0.023245,0.023249,0.497655,23427.1332,23208.4377,46635.5709


Therefore, in chunk 0, we have features for `1001` order-book states,given that each update corresponds to a change within 100 miliseconds, `chunk 0` corresponds to an interval of roughly **100 seconds** (or 1min40s)

In [57]:
features_df.shape[0]

1001

### **Applying the same logic to obtain the entire Features-DataFrame across all `chunks`**

Understanding the Process:

**`Step 1`**: Defining the `extract_features` functions to collect the relevant features from each order book state. Since the Orderbook is defined as a class in the `fastlob` package, we just create an object and collect its attributes (which correspond to the features).

**`Step 2`**: Iterate through each chunk (there are 53 unique chunks in the ETHBTC dataset):
- Extract the current order book snapshot for the respective chunk
- Initialize the O.B. dictionary containing tuples of `(price, amount)` on both sides of ask & bid
- Some chunks (e.g. chunk 1) are empty, so we simply skip them
- Initialize the order book for the underlying chunk
- Load the Updates of the current chunk
- Iterate through each update, apply it to the order book state and **extract the features**

**`Step 3`**: Create a Polars `DataFrame` to store the features of each updated order-book state, as well as the `chunk_id` column that simply represents which chunk these "observations" belong to.

**`Step 4`**: Repeat the above steps for every chunk in the original `df` (there should be around 53 unique chunks), and concatenate all individual feature-dataframes into one new dataset

In [58]:
all_features_dfs = []

unique_chunks = df['chunk'].unique().sort()

# The extract_features function (same as in the chunk 0 example)
def extract_features(lob_obj):
    return {
        'spread': lob_obj.spread(),
        'midprice': lob_obj.midprice(),
        'weighted midprice': lob_obj.weighted_midprice(),
        'imbalance': lob_obj.imbalance(),
        'asks_volume': lob_obj.asks_volume(),
        'bids_volume': lob_obj.bids_volume(),
        'total volume': lob_obj.total_volume()
    }

for chunk_id in unique_chunks:
    print(f"Processing Chunk: {chunk_id}")

    # 1. Get snapshot for the current chunk
    current_snapshot_df = df.filter((pl.col('snapshot') == True) & (pl.col('chunk') == chunk_id))

    # Prepare snapshot dictionary 'sc_chunk'
    sc_chunk = {'bids': [], 'asks': []}
    for x in current_snapshot_df.filter(pl.col('side') == 'bid').iter_rows():
        price, amount = x[2], x[3]
        sc_chunk['bids'].append((price, amount))
    for x in current_snapshot_df.filter(pl.col('side') == 'ask').iter_rows():
        price, amount = x[2], x[3]
        sc_chunk['asks'].append((price, amount))

    # If snapshot is empty, skip the chunk
    if not sc_chunk['bids'] and not sc_chunk['asks']:
        print(f"  Skipping chunk {chunk_id}: No snapshot data found.")
        continue

    # Initialize Orderbook for the current chunk
    lob_chunk = Orderbook.from_snapshot(sc_chunk, start=True)

    # 2. Get updates for the current chunk
    current_updates_df = df.filter((pl.col('snapshot') == False) & (pl.col('chunk') == chunk_id))

    # Prepare 'U_chunk' list of update dictionaries
    U_chunk = []
    if not current_updates_df.is_empty():
        max_update_id_chunk = current_updates_df['update_id'].max()
        for update_id in range(1, max_update_id_chunk + 1):
            update_data = current_updates_df.filter(pl.col('update_id') == update_id)
            u = {'bids': [], 'asks': []}
            for x in update_data.filter(pl.col('side') == 'bid').iter_rows():
                price, amount = x[2], x[3]
                u['bids'].append((price, amount))
            for x in update_data.filter(pl.col('side') == 'ask').iter_rows():
                price, amount = x[2], x[3]
                u['asks'].append((price, amount))
            U_chunk.append(u)

    # 3. Extract features for the current chunk
    features_data_chunk = []

    # Add features for the initial snapshot state
    initial_features_chunk = extract_features(lob_chunk)
    features_data_chunk.append(initial_features_chunk)

    # Iterate through each update, apply it, and extract features
    for i, update_dict in enumerate(U_chunk):
        # Check if the lob_chunk has been stopped or is invalid before loading updates
        if not lob_chunk.is_running():
            print(f"  Orderbook for chunk {chunk_id} stopped unexpectedly. Skipping remaining updates.")
            break

        lob_chunk.load_updates([update_dict])
        lob_chunk.step()
        current_features_chunk = extract_features(lob_chunk)
        features_data_chunk.append(current_features_chunk)

    # Create a Polars DataFrame for the current chunk's features
    features_df_chunk = pl.DataFrame(features_data_chunk)

    # Add chunk_id column to identify which chunk these features belong to
    features_df_chunk = features_df_chunk.with_columns(pl.lit(chunk_id).alias('chunk_id'))

    all_features_dfs.append(features_df_chunk)

# Concatenate all individual chunk feature DataFrames into one large DataFrame
if all_features_dfs:
    features_df_all_chunks = pl.concat(all_features_dfs)
    # Cast columns to Float64 for consistent decimal handling
    features_df_all_chunks = features_df_all_chunks.with_columns(
        pl.col('spread').cast(pl.Float64),
        pl.col('midprice').cast(pl.Float64),
        pl.col('weighted midprice').cast(pl.Float64),
        pl.col('imbalance').cast(pl.Float64),
        pl.col('asks_volume').cast(pl.Float64),
        pl.col('bids_volume').cast(pl.Float64),
        pl.col('total volume').cast(pl.Float64)
    )
    print("\nFeatures DataFrame for all chunks created successfully!")
    display(features_df_all_chunks.head())
    display(features_df_all_chunks.tail())
else:
    print("No features data generated for any chunk.")

Processing Chunk: 0
Processing Chunk: 2
Processing Chunk: 3
Processing Chunk: 4
Processing Chunk: 5
Processing Chunk: 6
Processing Chunk: 7
Processing Chunk: 8
Processing Chunk: 9
Processing Chunk: 10
Processing Chunk: 11
Processing Chunk: 12
Processing Chunk: 13
Processing Chunk: 14
Processing Chunk: 15
Processing Chunk: 16
Processing Chunk: 17
Processing Chunk: 18
Processing Chunk: 19
Processing Chunk: 20
Processing Chunk: 21
Processing Chunk: 22
Processing Chunk: 23
Processing Chunk: 24
Processing Chunk: 25
Processing Chunk: 26
Processing Chunk: 27
Processing Chunk: 28
Processing Chunk: 29
Processing Chunk: 30
Processing Chunk: 31
Processing Chunk: 32
Processing Chunk: 33
Processing Chunk: 34
Processing Chunk: 35
Processing Chunk: 36
Processing Chunk: 37
Processing Chunk: 38
Processing Chunk: 40
Processing Chunk: 41
Processing Chunk: 42
Processing Chunk: 43
Processing Chunk: 44
Processing Chunk: 45
Processing Chunk: 46
Processing Chunk: 47
Processing Chunk: 48
Processing Chunk: 49
P

spread,midprice,weighted midprice,imbalance,asks_volume,bids_volume,total volume,chunk_id
f64,f64,f64,f64,f64,f64,f64,i32
0.00001,0.023245,0.023249,0.497507,23425.7576,23193.3354,46619.093,0
0.00001,0.023245,0.023249,0.497507,23425.7576,23193.3354,46619.093,0
0.00001,0.023245,0.023249,0.49753,23423.8177,23193.5238,46617.3415,0
0.00001,0.023245,0.023249,0.497589,23423.8177,23199.008,46622.8257,0
0.00001,0.023245,0.023249,0.497655,23427.1332,23208.4377,46635.5709,0


spread,midprice,weighted midprice,imbalance,asks_volume,bids_volume,total volume,chunk_id
f64,f64,f64,f64,f64,f64,f64,i32
0.00001,0.022205,0.022201,0.540247,23456.8596,27563.6792,51020.5388,53
0.00001,0.022205,0.022201,0.540249,23456.8595,27563.9482,51020.8077,53
0.00001,0.022205,0.022201,0.540259,23456.8595,27565.03,51021.8895,53
0.00001,0.022205,0.022201,0.540259,23456.8156,27565.03,51021.8456,53
0.00001,0.022205,0.022201,0.540259,23456.8595,27565.03,51021.8895,53


For instance, in `chunk10` price has only moved 10 times, whilst in `chunk0` it has moved 83 times.

In [76]:
chunk10 = features_df_all_chunks.filter(pl.col('chunk_id') == 10)
chunk0 = features_df_all_chunks.filter(pl.col('chunk_id') == 0)
len(chunk10['midprice'].unique()), len(chunk0['midprice'].unique())

(17, 83)

Download the Final DataFrame for **`ETHBTC`**:

In [77]:
features_df_all_chunks.write_csv('/content/drive/MyDrive/Colab_Notebooks/Beta_Sigma/ETHBTCFinal.csv')